# Aggregation Contrast Insights

Run `scripts/build_paper_tables.py` and `scripts/find_aggregation_contrasts.py` first. This notebook reads the selected contrast cases and Pareto-front summary to inspect when EntropicOT or Additive clearly wins, trails, or lands on the Pareto front.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mdu.eval.paper_tables import slugify

PAPER_TABLES_DIR = ROOT / "resources/paper_tables"
OUTPUT_DIR = PAPER_TABLES_DIR / "aggregation_contrasts"
PARETO_PATH = PAPER_TABLES_DIR / "pareto_summary.csv"

cases = pd.read_csv(OUTPUT_DIR / "contrast_cases.csv")
summary_by_winner = pd.read_csv(OUTPUT_DIR / "summary_by_winner.csv")
failure_counts = pd.read_csv(OUTPUT_DIR / "failure_counts.csv")
summary_by_problem = pd.read_csv(OUTPUT_DIR / "summary_by_problem.csv")

if PARETO_PATH.exists():
    pareto = pd.read_csv(PARETO_PATH)
else:
    pareto = pd.DataFrame()
    print(f"Pareto summary not found: {PARETO_PATH}")
    print("Run scripts/build_paper_tables.py first to create it.")

print(f"Selected cases: {len(cases)}")
print(f"Pareto rows: {len(pareto)}")
cases.head()

## Pareto-front performance by aggregation

In [ ]:
if pareto.empty:
    print("No Pareto summary loaded.")
else:
    display(pareto.sort_values("pareto_percentage", ascending=False).head(30))

    ax = pareto.groupby("aggregation")["pareto_percentage"].mean().sort_values(ascending=False).plot(
        kind="bar",
        figsize=(7, 4),
    )
    ax.set_ylabel("mean Pareto-front percentage")
    ax.set_title("Average Pareto-front presence by aggregation")
    plt.tight_layout()

## Which aggregation wins the Pareto comparison per composition?

In [ ]:
if pareto.empty:
    print("No Pareto summary loaded.")
else:
    pareto_winners = (
        pareto.sort_values("pareto_percentage", ascending=False)
        .drop_duplicates("composition")
        .sort_values(["aggregation", "pareto_percentage"], ascending=[True, False])
    )
    display(pareto_winners[["composition", "aggregation", "pareto_percentage", "average_pareto_depth"]])

    ax = pareto_winners["aggregation"].value_counts().reindex(
        ["EntropicOT", "Additive"],
        fill_value=0,
    ).plot(kind="bar", figsize=(7, 4))
    ax.set_ylabel("number of compositions")
    ax.set_title("Best Pareto aggregation per composition")
    plt.tight_layout()


## Where does Additive improve over EntropicOT on Pareto percentage?


In [ ]:
if pareto.empty:
    print("No Pareto summary loaded.")
else:
    pareto_wide = pareto.pivot_table(
        index="composition",
        columns="aggregation",
        values="pareto_percentage",
        aggfunc="first",
    )
    for aggregation in ["Additive"]:
        if aggregation in pareto_wide.columns and "EntropicOT" in pareto_wide.columns:
            pareto_wide[f"{aggregation}_minus_EntropicOT"] = (
                pareto_wide[aggregation] - pareto_wide["EntropicOT"]
            )

    delta_cols = [col for col in ["Additive_minus_EntropicOT"] if col in pareto_wide.columns]
    if delta_cols:
        display(pareto_wide.sort_values(delta_cols, ascending=False).head(30))

        ax = pareto_wide[delta_cols].mean().sort_values(ascending=False).plot(kind="bar", figsize=(7, 4))
        ax.axhline(0.0, color="black", linewidth=1)
        ax.set_ylabel("mean Pareto percentage delta")
        ax.set_title("Baseline delta against EntropicOT")
        plt.tight_layout()
    else:
        display(pareto_wide.head(30))


## Pareto depth: lower is better

In [ ]:
if pareto.empty:
    print("No Pareto summary loaded.")
else:
    depth_summary = pareto.groupby("aggregation", as_index=False).agg(
        mean_depth=("average_pareto_depth", "mean"),
        median_depth=("median_pareto_depth", "mean"),
        mean_pareto_percentage=("pareto_percentage", "mean"),
    ).sort_values("mean_depth")
    display(depth_summary)

    ax = depth_summary.set_index("aggregation")["mean_depth"].plot(kind="bar", figsize=(7, 4))
    ax.set_ylabel("mean Pareto depth")
    ax.set_title("Average Pareto depth by aggregation")
    plt.tight_layout()

## How often does each aggregation clearly win?

In [ ]:
display(summary_by_winner)

if not summary_by_winner.empty:
    ax = summary_by_winner.pivot_table(
        index="winner",
        columns="problem_type",
        values="n_cases",
        aggfunc="sum",
        fill_value=0,
    ).plot(kind="bar", figsize=(9, 4))
    ax.set_ylabel("selected cases")
    ax.set_title("Clear wins by aggregation and task")
    plt.tight_layout()

## Which aggregations break when another one wins?

In [ ]:
display(failure_counts)

if not failure_counts.empty:
    ax = failure_counts.pivot_table(
        index="broken_aggregation",
        columns="winner",
        values="n_cases",
        aggfunc="sum",
        fill_value=0,
    ).plot(kind="bar", figsize=(9, 4))
    ax.set_ylabel("times trailing the winner")
    ax.set_title("Failure counts conditioned on the winner")
    plt.tight_layout()

## Strongest individual contrasts

In [ ]:
columns = [
    "composition",
    "problem_type",
    "ind_dataset",
    "eval",
    "winner",
    "winner_score",
    "second_best",
    "second_best_score",
    "worst",
    "worst_score",
    "gap_to_second",
    "gap_to_worst",
    "broken_aggregations",
]
display(cases.sort_values("gap_to_second", ascending=False)[columns].head(30))

## Drill down into one selected composition table

In [ ]:
if cases.empty:
    print("No contrast cases found. Try lowering --min_gap or --min_broken.")
else:
    case = cases.sort_values("gap_to_second", ascending=False).iloc[0]
    key = f"{slugify(case['composition'])}_{case['problem_type']}"
    table_path = OUTPUT_DIR / "selected_tables" / f"{key}_mean.csv"
    print(f"Composition: {case['composition']}")
    print(f"Problem type: {case['problem_type']}")
    print(f"Winner: {case['winner']}")
    print(table_path)
    display(pd.read_csv(table_path))

## Browse saved selected tables

In [ ]:
table_files = sorted((OUTPUT_DIR / "selected_tables").glob("*_mean.csv"))
pd.DataFrame({"table": [path.name for path in table_files]}).head(50)